# 03 — Prepare Model Data

Joins county-level mean forecast bias with ACS demographic data and
Koppen-Geiger climate classification. Exports a clean, analysis-ready
parquet for the BYM2 model.

**Inputs:**
- Pipeline outputs (via `PipelineDB`): `ifs_bias`, `koppen`
- ACS parquet: `{ACS_DIR}/acs_5yr_{ACS_YEAR}/acs_5yr_{ACS_YEAR}_county.parquet`

**Output:** `analysis/data/model_input.parquet`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..'))
sys.path.insert(0, os.path.join('..', 'scripts'))

import numpy as np
import pandas as pd

from nwp_census_eval.db import PipelineDB
from config import ACS_DIR, ACS_YEAR, ACS_LEVEL

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
# 1. Mean bias per county per lead time
with PipelineDB() as db:
    df_bias = db.query("""
        SELECT
            geo_id,
            lead_time,
            DAYOFYEAR(valid_time) AS day_of_year,
            AVG(bias)              AS mean_bias,
            COUNT(*)               AS n_obs
        FROM ifs_bias
        GROUP BY geo_id, lead_time, DAYOFYEAR(valid_time)
    """)

    # Koppen classification
    if 'koppen' in db.registered_views():
        df_koppen = db.query('SELECT geo_id, category_1 AS koppen_class FROM koppen')
    else:
        df_koppen = pd.DataFrame(columns=['geo_id', 'koppen_class'])

print(f'Bias rows: {len(df_bias):,} | Koppen: {len(df_koppen):,} counties')

In [ ]:
# 2. ACS demographics
acs_path = os.path.join(ACS_DIR, f'acs_5yr_{ACS_YEAR}', f'acs_5yr_{ACS_YEAR}_{ACS_LEVEL}.parquet')
df_acs = pd.read_parquet(acs_path)
print(f'ACS: {df_acs.shape} — columns: {list(df_acs.columns[:10])} ...')

In [ ]:
# 2b. Derive vulnerability indices from raw ACS columns
def _pct(num, denom, scale=100.0):
    """Safe percentage: returns NaN if denom is 0 or missing."""
    return np.where(denom > 0, num / denom * scale, np.nan)

# ---- Age structure (B01001) ----
total_pop = df_acs["B01001_001E"]

# Elderly 65+ = male rows 020-025 + female rows 044-049
male_65plus   = df_acs[[f"B01001_{r:03d}E" for r in range(20, 26)]].sum(axis=1)
female_65plus = df_acs[[f"B01001_{r:03d}E" for r in range(44, 50)]].sum(axis=1)
df_acs["demo_pct_elderly"] = _pct(male_65plus + female_65plus, total_pop)

# Children <5 = B01001_003E (male 1-4) + B01001_027E (female 1-4)
children_u5 = df_acs["B01001_003E"] + df_acs["B01001_027E"]
df_acs["demo_pct_children"] = _pct(children_u5, total_pop)

# ---- Race / ethnicity (B03002) ----
total_b03 = df_acs["B03002_001E"]
df_acs["demo_pct_nonwhite"]  = _pct(total_b03 - df_acs["B03002_003E"], total_b03)
df_acs["demo_pct_black"]     = _pct(df_acs["B03002_004E"], total_b03)
df_acs["demo_pct_hispanic"]  = _pct(df_acs["B03002_012E"], total_b03)

# ---- Education (B15003) — pop 25+, no HS diploma ----
edu_total  = df_acs["B15003_001E"]
no_hs      = df_acs[[f"B15003_{r:03d}E" for r in range(2, 17)]].sum(axis=1)
df_acs["demo_pct_no_hs"] = _pct(no_hs, edu_total)

# ---- Poverty (B17001) ----
pov_total  = df_acs["B17001_001E"]
below_pov  = df_acs["B17001_002E"]
df_acs["demo_pct_poverty"] = _pct(below_pov, pov_total)

# ---- Income (B19013, B19083) ----
df_acs["demo_log_income"] = np.log1p(df_acs["B19013_001E"].clip(lower=0))
df_acs["demo_gini"]       = df_acs["B19083_001E"]

# ---- Housing tenure (B25003) — % renters ----
tenure_total = df_acs["B25003_001E"]
renters      = df_acs["B25003_003E"]
df_acs["demo_pct_renters"] = _pct(renters, tenure_total)

# ---- Housing age (B25035) — median year built ----
df_acs["demo_median_year_built"] = df_acs["B25035_001E"]

# ---- Disability (B18101) ----
male_disabled   = df_acs[[f"B18101_{r:03d}E" for r in [4,7,10,13,16,19]]].sum(axis=1)
female_disabled = df_acs[[f"B18101_{r:03d}E" for r in [23,26,29,32,35,38]]].sum(axis=1)
dis_total = df_acs["B18101_001E"]
df_acs["demo_pct_disabled"] = _pct(male_disabled + female_disabled, dis_total)

# ---- Health insurance (B27001) — % uninsured ----
ins_total    = df_acs["B27001_001E"]
unins_male   = df_acs[[f"B27001_{r:03d}E" for r in [5,8,11,14,17,20,23,26,29]]].sum(axis=1)
unins_female = df_acs[[f"B27001_{r:03d}E" for r in [33,36,39,42,45,48,51,54,57]]].sum(axis=1)
df_acs["demo_pct_uninsured"] = _pct(unins_male + unins_female, ins_total)

# ---- Internet (B28002) — % without broadband ----
inet_total   = df_acs["B28002_001E"]
no_broadband = df_acs["B28002_013E"]
df_acs["demo_pct_no_broadband"] = _pct(no_broadband, inet_total)

# ---- Language isolation (C16002) — % linguistically isolated HHs ----
lang_total    = df_acs["C16002_001E"]
lang_isolated = df_acs[[f"C16002_{r:03d}E" for r in [4,7,10,13]]].sum(axis=1)
df_acs["demo_pct_lang_isolated"] = _pct(lang_isolated, lang_total)

demo_cols = [c for c in df_acs.columns if c.startswith("demo_")]
print(f"Derived {len(demo_cols)} vulnerability indices: {demo_cols}")

In [ ]:
# 3. Join all together
# Standardise geo_id format (leading zeros for 5-digit FIPS)
df_bias['geo_id'] = df_bias['geo_id'].astype(str).str.zfill(5)
df_acs['geo_id'] = df_acs['geo_id'].astype(str).str.zfill(5) if 'geo_id' in df_acs.columns else df_acs['GEOID'].astype(str).str.zfill(5)

df = df_bias.merge(df_koppen, on='geo_id', how='left')
df = df.merge(df_acs, on='geo_id', how='left')
print(f'Joined: {len(df):,} rows | {df.shape[1]} columns')

In [ ]:
# 4. Save model input
out_path = os.path.join(DATA_DIR, 'model_input.parquet')
df.to_parquet(out_path, index=False)
print(f'Saved {len(df):,} rows → {out_path}')